# Standardizing Strings (`str_strip_whitespace`, `str_to_lower`)

When dealing with user input or third-party APIs, strings are frequently messy. Users might accidentally add trailing spaces when copying and pasting (e.g., `"Python "`), or they might use inconsistent casing (`"user@Email.com"` vs `"user@email.com"`).

Instead of manually writing `.strip()` and `.lower()` throughout your application logic, you can configure Pydantic to automatically clean all string fields across the entire model during instantiation.

## 1. Stripping Whitespace (`str_strip_whitespace`)

By setting `str_strip_whitespace=True` in the `model_config`, Pydantic will automatically run the equivalent of Python's `.strip()` on every string field, removing leading and trailing spaces, tabs (`\t`), and newlines (`\n`).

```python
from pydantic import BaseModel, ConfigDict

class CleanStringModel(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)
    
    first_name: str
    last_name: str

# Messy input with tabs, spaces, and newlines
raw_data = {
    "first_name": "   \tGuido  ",
    "last_name": "\nvan Rossum \n"
}

m1 = CleanStringModel(**raw_data)

print(repr(m1.first_name)) # 'Guido'
print(repr(m1.last_name))  # 'van Rossum'

```

*Note: This only strips leading and trailing whitespace. It does not affect spaces *between* words (e.g., `"van Rossum"` remains intact).*

## 2. Enforcing Case (`str_to_lower` / `str_to_upper`)

You can force all strings in a model to a specific casing. This is extremely useful for things like email addresses, promo codes, or standardized database keys.

```python
class EmailModel(BaseModel):
    model_config = ConfigDict(
        str_strip_whitespace=True, 
        str_to_lower=True
    )
    
    email: str

m2 = EmailModel(email="   Admin@EXAMPLE.com  ")
print(repr(m2.email)) # 'admin@example.com'

```

* You can also use `str_to_upper=True` to force everything to uppercase (useful for things like Country Codes or License Plates).
* **Important:** If you accidentally define *both* `str_to_lower=True` and `str_to_upper=True`, Pydantic does not throw an error; it silently prioritizes `str_to_lower`. You should obviously avoid defining both.


### Interview Preparation: Standardizing Strings

<details>
<summary><b>1. You are building an API endpoint for a user registration form. Users often paste their email addresses with accidental trailing spaces, and some use capital letters. How do you ensure your database only stores clean, lowercase emails without writing any manual pre-processing code?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
I would handle this entirely within the Pydantic schema using the `model_config` attribute. <br><br>
By setting `model_config = ConfigDict(str_strip_whitespace=True, str_to_lower=True)`, Pydantic will automatically intercept the incoming JSON string, strip any leading or trailing spaces (or newlines/tabs), and convert the entire string to lowercase before the model instance is fully created. I can then safely pass the validated object directly to my database ORM.
</details>

<details>
<summary><b>2. If you set `str_strip_whitespace=True` on a model, and a user inputs `"John   Doe"`, will Pydantic convert that to `"John Doe"`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
No, it will remain `"John   Doe"`. <br><br>
The `str_strip_whitespace` configuration relies on standard stripping logic (like Python's `.strip()` method). It only targets leading (beginning) and trailing (end) whitespace. It does not attempt to compress or normalize whitespace that exists *between* characters within the string itself. To do that, you would need to write a custom `@field_validator`.
</details>

<details>
<summary><b>3. Is there a scenario where setting `str_to_lower=True` at the *model configuration* level is a bad idea? How would you resolve it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It is a bad idea if your model contains mixed string fields where casing matters for some, but not for others. <br><br>
For example, if my model has `email` (which should be lowercase) and `password` (which is highly case-sensitive), setting `str_to_lower=True` at the model config level will convert the password to lowercase, completely breaking the user's login. <br><br>
In that scenario, you cannot use the model-level config. You must instead handle it at the field level, either by using a custom `@field_validator` on the email field, or by using Pydantic's `StringConstraints` via `typing.Annotated`.
</details>